In [2]:
import pandas as pd
df = pd.read_csv("raw_data.csv")
df

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2,2.0,4.0,NaN,UNKNOWN,2023-08-30
9996,TXN_9659401,NaN,3,NaN,3.0,Digital Wallet,NaN,2023-06-02
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3,NaN,3.0,Digital Wallet,NaN,2023-12-02


In [10]:
df.info()
print("-----")
print(df.isnull().sum())
print("-----")
print("Duplicates:", df.duplicated().sum())

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    10000 non-null  str  
 1   Item              9667 non-null   str  
 2   Quantity          9862 non-null   str  
 3   Price Per Unit    9821 non-null   str  
 4   Total Spent       9827 non-null   str  
 5   Payment Method    7421 non-null   str  
 6   Location          6735 non-null   str  
 7   Transaction Date  9841 non-null   str  
dtypes: str(8)
memory usage: 1.1 MB
-----
Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64
-----
Duplicates: 0


In [11]:
import numpy as np

rows_before = len(df)
df_clean = df.copy()
df_clean = df_clean.replace(["ERROR", "UNKNOWN"], np.nan)
print("Rows before:", rows_before)
print(df_clean.isnull().sum())

Rows before: 10000
Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64


In [12]:
df_clean["Quantity"] = pd.to_numeric(df_clean["Quantity"], errors="coerce")
df_clean["Price Per Unit"] = pd.to_numeric(df_clean["Price Per Unit"], errors="coerce")
df_clean["Total Spent"] = pd.to_numeric(df_clean["Total Spent"], errors="coerce")
df_clean["Transaction Date"] = pd.to_datetime(df_clean["Transaction Date"], errors="coerce")
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    10000 non-null  str           
 1   Item              9031 non-null   str           
 2   Quantity          9521 non-null   float64       
 3   Price Per Unit    9467 non-null   float64       
 4   Total Spent       9498 non-null   float64       
 5   Payment Method    6822 non-null   str           
 6   Location          6039 non-null   str           
 7   Transaction Date  9540 non-null   datetime64[us]
dtypes: datetime64[us](1), float64(3), str(4)
memory usage: 897.5 KB


In [15]:
df_clean["Price Per Unit"] = df_clean["Price Per Unit"].fillna(df_clean["Total Spent"] / df_clean["Quantity"])
df_clean["Quantity"] = df_clean["Quantity"].fillna(df_clean["Total Spent"] / df_clean["Price Per Unit"])
df_clean["Total Spent"] = df_clean["Total Spent"].fillna(df_clean["Quantity"] * df_clean["Price Per Unit"])

df_clean = df_clean.dropna(subset=["Quantity", "Price Per Unit", "Total Spent"])
df_clean["Quantity"] = df_clean["Quantity"].astype(int)

In [16]:
for col in ["Item", "Payment Method", "Location"]:
    df_clean[col] = df_clean[col].fillna("Unknown")

print("Rows before:", rows_before)
print("Rows after:", len(df_clean))
print(df_clean.isnull().sum())
df_clean.head()

Rows before: 10000
Rows after: 9942
Transaction ID        0
Item                  0
Quantity              0
Price Per Unit        0
Total Spent           0
Payment Method        0
Location              0
Transaction Date    457
dtype: int64


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,4.0,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,Unknown,Unknown,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [17]:
df_clean.to_csv("cleaned_data.csv", index=False)
print("Saved! Rows:", len(df_clean))

Saved! Rows: 9942
